# 👋 Hello World — pi.dev (`@earendil-works/pi-ai`)

First real step in exploring [**pi**](https://github.com/earendil-works/pi) — an open-source, multi-provider LLM API and agent framework. Goal: import the library, pick a model, and get one response back.

We use **`@earendil-works/pi-ai`**, the unified LLM API. Later notebooks build actual agents on top of it.

> **Kernel:** **Deno** (pi.dev is an ESM-only TypeScript package; the Deno kernel runs it natively with top-level await). See `../README.md`.

## 1. Load your API key from a `.env` outside the repo

`loadEnvUp()` walks **up** the directory tree from the current folder, finds the first `.env`, and imports its variables into this session. Keep your key on disk but **outside the git repo** (e.g. in a parent folder), so it can never be committed:

```
# ../.env  (one directory above the repo)
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import { loadEnvUp } from "playground/env";

const env = await loadEnvUp();
console.log(env.path ? `Loaded ${env.loaded.length} var(s) from ${env.path}` : "No .env found up-tree.");

In [ ]:
// Register every built-in provider into a Models collection.
import { type Context } from "@earendil-works/pi-ai";
import { builtinModels } from "@earendil-works/pi-ai/providers/all";

const models = builtinModels();
console.log("pi-ai loaded ✔");

In [ ]:
// Pick a model based on whichever provider key is in the environment.
// Anthropic uses Haiku here. Add your own line for other providers.
const candidates: Array<[string, string, string]> = [
  ["ANTHROPIC_API_KEY", "anthropic", "claude-haiku-4-5"],
  ["OPENAI_API_KEY",    "openai",    "gpt-4o-mini"],
  ["GEMINI_API_KEY",    "google",    "gemini-2.5-flash"],
];

const picked = candidates.find(([envVar]) => Deno.env.get(envVar));
if (!picked) {
  throw new Error("No provider API key found. Add one to your .env (e.g. ANTHROPIC_API_KEY) and re-run cell 1.");
}

const [envVar, provider, modelId] = picked;
const model = models.getModel(provider, modelId);
if (!model) throw new Error(`Model ${provider}/${modelId} not found in the catalog.`);

console.log(`Using ${provider}/${modelId} (auth via ${envVar}).`);

In [ ]:
// Hello world: send one message and print the reply.
const context: Context = {
  systemPrompt: "You are a friendly assistant. Keep answers to one short sentence.",
  messages: [
    { role: "user", content: "Say hello and confirm you are running via the pi.dev library.", timestamp: Date.now() },
  ],
};

const response = await models.completeSimple(model, context);

for (const block of response.content) {
  if (block.type === "text") console.log("🤖", block.text);
}

console.log(
  `\ntokens: ${response.usage.input} in / ${response.usage.output} out` +
  `  |  cost: $${response.usage.cost.total.toFixed(6)}`,
);

## ✅ What just happened

1. `loadEnvUp()` pulled your API key from a `.env` kept outside the repo.
2. `builtinModels()` gave a `Models` collection with every provider registered.
3. `models.getModel(provider, id)` resolved a concrete model (Anthropic → **Haiku**).
4. `models.completeSimple(model, context)` ran one request, resolving auth from the environment, and returned message `content` blocks plus token/cost `usage`.

## Next steps to explore

- **Streaming**: swap `completeSimple` for `models.streamSimple(model, context)` and iterate the events (`text_delta`, `toolcall_end`, …).
- **Tools**: define a `Tool` with a TypeBox schema and handle `toolCall` blocks — the foundation of an agent loop.
- **Agents**: use `@earendil-works/pi-agent-core`'s `Agent` class, which wraps the tool-calling loop for you.